# Figure: build + traversal scaling vs. N

Loads `results/differentiability/scaling.json` (produced by `bench/differentiability/scaling.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

# Resolve the results directory whether the notebook is run from its own
# folder or from the repo root.
_here = Path.cwd()
_candidates = [
    _here / "results" / "differentiability",
    _here.parents[1] / "results" / "differentiability",
]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
PALETTE = {"radix": "#4C72B0", "octree": "#DD8452", "kdtree": "#55A868"}
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "scaling.json").read_text())
meta = data["metadata"]
backends = data["params"]["backends"]
records = data["records"]
ns = [r["n"] for r in records]
print(f"device={meta['device_kind']} backend={meta['backend']} "
      f"jax={meta['jax_version']} commit={meta.get('git_commit')}")


In [ ]:
fig, (ax_b, ax_t) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

for be in backends:
    build = [r["backends"][be]["build"]["min_s"] * 1e3 for r in records]
    trav = [r["backends"][be]["traverse"]["min_s"] * 1e3 for r in records]
    ax_b.plot(ns, build, "o-", color=PALETTE.get(be), label=be)
    ax_t.plot(ns, trav, "o-", color=PALETTE.get(be), label=be)

# scipy cKDTree CPU baseline (build), if present.
if all("scipy_ckdtree" in r for r in records):
    sci = [r["scipy_ckdtree"]["build_min_s"] * 1e3 for r in records]
    ax_b.plot(ns, sci, "s--", color=BASELINE, label="scipy cKDTree (CPU)")

for ax, title in ((ax_b, "Tree build"), (ax_t, "Dual-tree traversal")):
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("N particles"); ax.set_ylabel("time [ms]")
    ax.set_title(title); ax.legend()
fig.suptitle(f"Scaling on {meta['device_kind']} ({meta['backend']})", fontsize=12)

out = FIGDIR / "fig_scaling.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
